# mDeBERTa-v3 NLI INT8 Quantization via OpenVINO SmoothQuant - source-conditioned grounding

**Author**: Konrad Jelen (kj)<br>
**Pipeline stage**: 0 - model preparation (quantize the NLI entailer behind the grounding axis)

`mDeBERTa-v3-base-mnli-xnli` is the NLI half of the **source-conditioned document-distance grounding axis** `D_grd`. In `d(A,B|S)` the grounding axis scores each summary statement's entailment against its top-3 reranked source statements (a joint premise); the document grounding signature is `(ungrounded, contradiction)` and `D_grd` is its Euclidean distance from the gold anchor. The entailer must run INT8 on CPU, but its disentangled attention produces activation outliers that naive INT8 collapses (ONNX-Runtime dynamic 0.35, static 0.29). SmoothQuant migrates the outliers into the weights so activation-INT8 survives.

This notebook owns a reproducible, grounding-calibrated quantization in-project:

1. Export `mDeBERTa-v3-base-mnli-xnli` to an OpenVINO IR (FP32)
2. Quantize to INT8 with NNCF SmoothQuant across an `alpha` sweep (outlier-migration strength)
3. Calibrate on **real (source top-3 joint premise, summary statement) pairs** drawn from the exec-summary fixtures - the exact input distribution `D_grd` sees, not a generic NLI corpus
4. Gate on **distance fidelity**: INT8-vs-FP parity on per-statement entailment, plus the per-fixture `D_grd` correlation and tier-ordering / ordinality / severity all preserved - the quality measure is "does INT8 reach the same source-conditioned distance verdict as FP", not the grounding-stack macro-F1 of the original recipe

The reranker builds the joint premises once on GPU (fixed across both NLI arms, so only NLI precision varies); the FP32/INT8 mDeBERTa scoring and the NNCF quantization run on CPU via OpenVINO.

**Output**: best INT8 SmoothQuant IR under `models/Q02-mdeberta-openvino-int8/`, fidelity metrics under `reports/Q02-mdeberta-quantization-metrics.json`.

## Environment and GPU selection

The reranker premise pass runs on the RTX 5000 Ada (FP16); the NNCF quantization and all OpenVINO NLI scoring run on CPU. GPU env set before importing torch.

In [1]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"          # CUDA indices match nvidia-smi
# resolve the freest GPU and pin it by UUID (portable - never a hardcoded index)
import subprocess as _sp
def _free_gpu_uuid():
    try:
        rows = _sp.check_output(
            ["nvidia-smi", "--query-gpu=uuid,memory.used,utilization.gpu", "--format=csv,noheader,nounits"],
            text=True).strip().splitlines()
        gpus = [(u.strip(), int(mu), int(ut)) for u, mu, ut in (r.split(",") for r in rows)]
        return min(gpus, key=lambda g: (g[2], g[1]))[0] if gpus else None
    except Exception:
        return None
_gpu = _free_gpu_uuid()
if _gpu:
    os.environ["CUDA_VISIBLE_DEVICES"] = _gpu                # RTX 5000 Ada (sm_89, 32 GB) - reranker only
os.environ["HF_HUB_OFFLINE"] = "1"                      # models already cached
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"        # no weight-download / loading bars

import torch

assert torch.cuda.is_available(), "GPU required for the reranker premise pass"
print(f"GPU: {torch.cuda.get_device_name(0)} | cap {torch.cuda.get_device_capability(0)} | torch {torch.__version__}")

GPU: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition | cap (12, 0) | torch 2.11.0+cu130


## Imports

Quantization toolchain (OpenVINO / NNCF via optimum-intel), the project segmenter, the FP reranker / NLI on torch, and rich/scipy for scoring and output.

In [2]:
get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")

# Standard library
import contextlib
import gc
import io
import json
import time
import warnings
from pathlib import Path

# Silence known third-party noise before the heavy imports pull in tqdm.auto
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# Data processing
import numpy as np
import scipy.special as sp
import datasets
from scipy.stats import pearsonr, spearmanr, rankdata

# OpenVINO / NNCF quantization (optimum-intel)
import openvino as ov
from optimum.intel import (OVModelForSequenceClassification, OVQuantizer,
                           OVConfig, OVQuantizationConfig)

# Transformers (FP reference + reranker)
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from transformers.utils import logging as hf_logging

# Project
from docdistance.encoders import Segmenter

# Rich console output
from rich import print as rprint

datasets.disable_progress_bars()                       # no dataset map() bars in cell output
hf_logging.disable_progress_bar()                      # no transformers "Loading weights" bars

## Reproducibility

Fixed seed - the calibration pairing is deterministic (every summary statement contributes one pair), so the seed only guards any incidental sampling.

In [3]:
SEED = 42
np.random.seed(SEED)

## Configuration

All knobs for the quantization run. `alpha` is SmoothQuant's migration strength (higher pushes more activation outliers into the weights - the lever for DeBERTa's severe outliers). The acceptance gate is distance fidelity: INT8-vs-FP parity on entailment, plus preservation of the source-conditioned verdict (tier ordering, severity regime, per-document ordinality no worse). The per-fixture `D_grd` Pearson is a supporting diagnostic.

In [4]:
# Models
MODEL_ID = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"   # DeBERTa-v2 NLI, entailment = label 0
RERANKER_ID = "BAAI/bge-reranker-v2-m3"                # cross-encoder relevance (FP, GPU, fixed)

# Source-conditioned grounding (matches the E02/E03 chain)
TOP_K = 3                                              # source statements fused into each joint premise

# SmoothQuant sweep
ALPHAS = [0.5, 0.7, 0.9]                               # outlier-migration strength
CALIB_MAXLEN = 256                                     # calibration sequence cap

# Acceptance (distance fidelity, not stack macro-F1)
TARGET_PARITY = 0.99                                   # INT8 vs FP32 pearson on entailment (the gate)
TARGET_DGRD_R = 0.99                                   # reference for the D_grd pearson diagnostic

# Fixtures - the E02 set: 11 executive summaries of one IBM AI-adoption article, three tiers
ROOT = Path.cwd().parent.parent                        # notebooks/model-quantization/ -> project root
BASE = ROOT / "data" / "interim" / "exec-summaries" / "ibm-ai-adoption"
SOURCE_FILE = BASE / "source" / "source-article.md"
SUMMARY_DIR = BASE / "summaries"
DOCS = [
    ("exec-summary-gold-opus-4-6.md",     "gold",   "gold"),   # reference anchor
    ("exec-summary-gold-2-opus-4-6.md",   "gold-2", "gold"),
    ("exec-summary-1-opus-4-6.md",        "v1",     "gold"),
    ("exec-summary-2-opus-4-6.md",        "v2",     "gold"),
    ("exec-summary-opus-4-6.md",          "opus",   "gold"),
    ("exec-summary-sonnet-4-6.md",        "sonnet", "gold"),
    ("exec-summary-haiku-4-5.md",         "haiku",  "gold"),
    ("exec-summary-adv1-a-sonnet-4-6.md", "adv1-a", "adv1"),    # Set 1 - info-loss
    ("exec-summary-adv1-b-sonnet-4-6.md", "adv1-b", "adv1"),
    ("exec-summary-adv2-a-haiku-4-5.md",  "adv2-a", "adv2"),    # Set 2 - info-noise
    ("exec-summary-adv2-b-haiku-4-5.md",  "adv2-b", "adv2"),
]
ANCHOR = "gold"
TIER = {l: t for _, l, t in DOCS}
TIERS = ["gold", "adv1", "adv2"]
TIER_NAME = {"gold": "gold (faithful)", "adv1": "Set 1 (info-loss)", "adv2": "Set 2 (info-noise)"}

# Paths (IRs under git-ignored models/, metrics under tracked reports/)
WORK = ROOT / "models" / "Q02-mdeberta-openvino-int8"
FP32_DIR = WORK / "fp32"
METRICS_PATH = ROOT / "reports" / "Q02-mdeberta-quantization-metrics.json"
WORK.mkdir(parents=True, exist_ok=True)

core = ov.Core()

rprint(f"""[bold cyan]Configuration[/bold cyan]
[dim]{"-" * 46}[/dim]
[bold]Models[/bold]
  NLI (quantized): [cyan]{MODEL_ID}[/cyan]
  Reranker (FP, fixed): [cyan]{RERANKER_ID}[/cyan]
  Joint premise: [yellow]top-{TOP_K}[/yellow] source per statement

[bold]SmoothQuant[/bold]
  Alphas: [yellow]{ALPHAS}[/yellow]  [dim](calib max_len {CALIB_MAXLEN})[/dim]

[bold]Acceptance - distance fidelity[/bold]
  Entailment parity (pearson): [yellow]>= {TARGET_PARITY}[/yellow]
  Verdict preserved FP -> INT8: [yellow]ordering + severity + violations no worse[/yellow]
  D_grd parity (pearson): [dim]diagnostic, reference >= {TARGET_DGRD_R}[/dim]

[bold]Fixtures[/bold]
  Documents: [yellow]{len(DOCS)}[/yellow] summaries + 1 source, [yellow]{len(TIERS)}[/yellow] tiers  [dim](anchor {ANCHOR})[/dim]

[bold]Runtime[/bold]
  Quantize + NLI: [green]CPU[/green] [dim](OpenVINO {ov.__version__})[/dim]
  Reranker: [green]{torch.cuda.get_device_name(0)}[/green] [dim](FP16)[/dim]
""")

Configuration
----------------------------------------------
Models
  NLI (quantized): MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
  Reranker (FP, fixed): BAAI/bge-reranker-v2-m3
  Joint premise: top-3 source per statement

SmoothQuant
  Alphas: [0.5, 0.7, 0.9]  (calib max_len 256)

Acceptance - distance fidelity
  Entailment parity (pearson): >= 0.99
  Verdict preserved FP -> INT8: ordering + severity + violations no worse
  D_grd parity (pearson): diagnostic, reference >= 0.99

Fixtures
  Documents: 11 summaries + 1 source, 3 tiers  (anchor gold)

Runtime
  Quantize + NLI: CPU (OpenVINO 2026.2.1-21919-ede283a88e3-releases/2026/2)
  Reranker: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition (FP16)

## Data loading and calibration set

Segment the source and the 11 summaries, score the FP reranker grid on GPU, fuse the top-3 source per statement into a joint premise, and collect every (joint premise, summary statement) pair. The reranker grids are computed once and fixed; the calibration set is exactly the input distribution the INT8 entailer will see in `D_grd`.

In [5]:
def body(path):
    """Strip markdown headers, keep the prose the segmenter scores."""
    return "\n".join(l for l in Path(path).read_text().splitlines() if not l.startswith("# ")).strip()


class TorchReranker:
    """FP16 cross-encoder relevance on GPU; sigmoid(logit) in [0, 1]."""
    def __init__(self, repo):
        self.tok = AutoTokenizer.from_pretrained(repo)
        self.m = AutoModelForSequenceClassification.from_pretrained(repo).to("cuda").to(torch.float16).eval()

    @torch.no_grad()
    def grid(self, x_texts, s_texts, bs=256):
        xs = [x for x in x_texts for _ in s_texts]
        ss = [s for _ in x_texts for s in s_texts]
        out = []
        for i in range(0, len(xs), bs):
            e = self.tok(xs[i:i + bs], ss[i:i + bs], padding=True, truncation=True,
                         max_length=256, return_tensors="pt").to("cuda")
            out.append(self.m(**e).logits.float().cpu().numpy())
        return sp.expit(np.concatenate(out, 0)).reshape(len(x_texts), len(s_texts))


with contextlib.redirect_stderr(io.StringIO()):        # mute model-load progress, keep cell output clean
    seg = Segmenter()
    nli_tok = AutoTokenizer.from_pretrained(MODEL_ID)
    reranker = TorchReranker(RERANKER_ID)

S_texts = seg.split(body(SOURCE_FILE))

# build joint premises + per-document spans into the flat (premise, hyp) arrays
premises, hyps, spans = [], [], []
t0 = time.perf_counter()
for fname, label, tier in DOCS:
    X = seg.split(body(SUMMARY_DIR / fname))
    R = reranker.grid(X, S_texts)
    start = len(premises)
    for i in range(len(X)):
        topk = np.argsort(R[i])[::-1][:TOP_K]
        premises.append(" ".join(S_texts[j] for j in topk))
        hyps.append(X[i])
    spans.append((label, tier, start, len(premises), R.max(1).copy()))

N_CALIB = len(premises)
rprint(f"[white]source[/white] [yellow]{len(S_texts)}[/yellow] stmts | "
       f"[white]calibration pairs[/white] [yellow]{N_CALIB}[/yellow] "
       f"[dim](joint-premise, summary-stmt; {time.perf_counter() - t0:.1f}s)[/dim]")

# calibration dataset for NNCF - exactly the inference pairing, padded for static calibration
_enc = nli_tok(premises, hyps, truncation=True, max_length=CALIB_MAXLEN, padding="max_length")
calib = datasets.Dataset.from_dict({"input_ids": _enc["input_ids"],
                                    "attention_mask": _enc["attention_mask"]})

source 70 stmts | calibration pairs 98 (joint-premise, summary-stmt; 5.9s)

## Helpers - OpenVINO scoring and the grounding metric

`ov_probs` scores entailment and contradiction on an OpenVINO IR over the fixed joint premises. The grounding helpers rebuild the canonical E03 axis: per-document signature `(mean(1-entail), mean(contra))`, `D_grd = ||sig - sig(anchor)||`, and the tier verdict (ordinality violations, severity direction).

In [6]:
# entailment / contradiction indices from the model config (config only, no weight load)
_cfg = AutoConfig.from_pretrained(MODEL_ID)
ENT = int([k for k, v in _cfg.id2label.items() if v.lower().startswith("entail")][0])
CON = int([k for k, v in _cfg.id2label.items() if v.lower().startswith("contradi")][0])
print(f"NLI labels: entail {ENT}, contradict {CON}")

gold_l = [l for _, l, t in DOCS if t == "gold"]
adv_l = [l for _, l, t in DOCS if t != "gold"]
set1 = [l for _, l, t in DOCS if t == "adv1"]
set2 = [l for _, l, t in DOCS if t == "adv2"]


def ov_probs(xml_path, bs=16):
    """Per-statement (entail, contra) over the fixed premises/hyps, on an OpenVINO IR."""
    cm = core.compile_model(core.read_model(str(xml_path)), "CPU", {"PERFORMANCE_HINT": "THROUGHPUT"})
    names = [i.get_any_name() for i in cm.inputs]
    out = []
    for i in range(0, len(premises), bs):
        e = nli_tok(premises[i:i + bs], hyps[i:i + bs], padding=True, truncation=True,
                    max_length=256, return_tensors="np")
        feed = {n: (e[n].astype(np.int64) if n in e else np.zeros_like(e["input_ids"]))
                for n in names}
        out.append(cm(feed)[cm.output(0)])
    P = sp.softmax(np.concatenate(out, 0), axis=1)
    return P[:, ENT], P[:, CON]


def signatures(ent, con):
    """Per-document grounding signature (mean ungrounded, mean contradiction)."""
    return {label: np.array([float(np.mean(1.0 - ent[a:b])), float(np.mean(con[a:b]))])
            for (label, _, a, b, _) in spans}


def d_grd(sig):
    """Canonical grounding axis: Euclidean distance of each signature from the gold anchor."""
    a = sig[ANCHOR]
    return {l: float(np.linalg.norm(sig[l] - a)) for l in TIER}


def tier_means(score):
    return {t: float(np.mean([score[l] for l in TIER if TIER[l] == t])) for t in TIERS}


def violations(score):
    """Ordinality: count (gold, adv) pairs where gold >= adv. 0 is clean."""
    return sum(1 for g in gold_l for a in adv_l if score[g] >= score[a])


def severity(score):
    """Set 2 vs Set 1 split direction; 'Set2>Set1' is the correct grounding severity."""
    s1 = [score[l] for l in set1]; s2 = [score[l] for l in set2]
    if min(s2) > max(s1): return "Set2>Set1"
    if min(s1) > max(s2): return "Set1>Set2"
    return "interleaved"


def parity(a, b):
    return dict(pearson=float(pearsonr(a, b)[0]), spearman=float(spearmanr(a, b)[0]),
                max_abs=float(np.abs(np.asarray(a) - np.asarray(b)).max()))

NLI labels: entail 0, contradict 2


## Export FP32 OpenVINO IR

One-time export of the PyTorch checkpoint to an OpenVINO IR. SmoothQuant quantizes from this, and the FP32 IR is the lossless parity reference (the un-quantized model on the same OpenVINO runtime, so INT8 is the only variable). OpenVINO 2026.2's torch frontend inserts FP16 casts that clash with DeBERTa's FP32 LayerNorm weights, so the IR is built via `ov.convert_model` with FP16 compression off, not `OVModel(export=True)`.

In [7]:
if not (FP32_DIR / "openvino_model.xml").exists():
    t = time.time()
    with contextlib.redirect_stderr(io.StringIO()):
        fp_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, torch_dtype=torch.float32).eval()
        ex = nli_tok("source premise", "summary hypothesis", return_tensors="pt", truncation=True, max_length=32)
        ov_fp = ov.convert_model(fp_model, example_input={"input_ids": ex["input_ids"],
                                                          "attention_mask": ex["attention_mask"]})
    FP32_DIR.mkdir(parents=True, exist_ok=True)
    ov.save_model(ov_fp, str(FP32_DIR / "openvino_model.xml"), compress_to_fp16=False)  # keep FP32
    fp_model.config.save_pretrained(FP32_DIR)
    nli_tok.save_pretrained(FP32_DIR)
    del fp_model, ov_fp; gc.collect()
    rprint(f"[green]exported FP32 IR[/green] in [yellow]{time.time() - t:.0f}s[/yellow]")
fp32_mb = (FP32_DIR / "openvino_model.bin").stat().st_size / 1e6

# FP32 reference scores (entailment, contradiction) over the fixed premises
ent_fp, con_fp = ov_probs(FP32_DIR / "openvino_model.xml")
sig_fp = signatures(ent_fp, con_fp)
dgrd_fp = d_grd(sig_fp)
rprint(f"FP32 IR size: [yellow]{fp32_mb:.0f} MB[/yellow] | "
       f"FP32 tier means D_grd: " +
       "  ".join(f"{TIER_NAME[t].split()[0]} [yellow]{v:.3f}[/yellow]"
                 for t, v in tier_means(dgrd_fp).items()))

FP32 IR size: 1151 MB | FP32 tier means D_grd: gold 0.228  Set 0.285  Set 0.625

## Quantize with SmoothQuant (alpha sweep)

For each `alpha`, run NNCF INT8 PTQ with SmoothQuant on the grounding calibration set and parity-test the entailment scores against the FP32 reference. NNCF recurses into DeBERTa's `If` subgraphs (the disentangled-attention control flow) - the step plain ONNX-Runtime SmoothQuant could not handle. Memory is freed between alphas.

In [8]:
def quantize_smoothquant(alpha, out_dir):
    model = OVModelForSequenceClassification.from_pretrained(FP32_DIR)
    quantizer = OVQuantizer.from_pretrained(model)
    cfg = OVConfig(quantization_config=OVQuantizationConfig(
        bits=8, smooth_quant_alpha=alpha, num_samples=N_CALIB))
    quantizer.quantize(calibration_dataset=calib, save_directory=out_dir, ov_config=cfg)
    del model, quantizer; gc.collect()


results = []
for alpha in ALPHAS:
    sd = WORK / f"sq_a{int(alpha * 100)}"
    t = time.time()
    if not (sd / "openvino_model.xml").exists():
        quantize_smoothquant(alpha, sd)
    ent_q, con_q = ov_probs(sd / "openvino_model.xml")
    p = parity(ent_q, ent_fp)
    size_mb = (sd / "openvino_model.bin").stat().st_size / 1e6
    results.append(dict(alpha=alpha, size_mb=size_mb, secs=time.time() - t,
                        dir=str(sd), ent=ent_q, con=con_q, **p))
    rprint(f"  alpha=[yellow]{alpha}[/yellow]  size=[cyan]{size_mb:.0f} MB[/cyan]  "
           f"entail pearson=[{'green' if p['pearson'] >= TARGET_PARITY else 'yellow'}]{p['pearson']:.4f}[/]  "
           f"max|d|={p['max_abs']:.3f}  [dim]({time.time() - t:.0f}s)[/dim]")
    gc.collect()

alpha=0.5  size=318 MB  entail pearson=0.9810  max|d|=0.338  (9s)

alpha=0.7  size=318 MB  entail pearson=0.9957  max|d|=0.168  (9s)

alpha=0.9  size=318 MB  entail pearson=0.9863  max|d|=0.345  (9s)

## Parity results

INT8 entailment parity against FP32 for each alpha. ONNX-Runtime baselines on the same model (no SmoothQuant): dynamic 0.35, static 0.29, mixed-precision 0.754, weight-only-4bit 0.90 - SmoothQuant should clear all of them and reach the 0.99 gate.

In [9]:
best = max(results, key=lambda r: r["pearson"])
rows = "\n".join(
    f"  alpha [yellow]{r['alpha']}[/yellow]: entail pearson "
    f"[{'green' if r['pearson'] >= TARGET_PARITY else 'yellow'}]{r['pearson']:.4f}[/]  "
    f"max|d| {r['max_abs']:.3f}  size [cyan]{r['size_mb']:.0f} MB[/cyan]"
    + ("  [green](best)[/green]" if r is best else "")
    for r in results)
rprint(f"""[bold cyan]SmoothQuant INT8 entailment parity vs FP32[/bold cyan]
[dim]{"-" * 56}[/dim]
{rows}
[dim]ORT baselines (no SmoothQuant): dynamic 0.35 / static 0.29 / mixed 0.754 / wo4bit 0.90[/dim]

Best: alpha [yellow]{best['alpha']}[/yellow] at [green]{best['pearson']:.4f}[/green], [cyan]{best['size_mb']:.0f} MB[/cyan]
""")

SmoothQuant INT8 entailment parity vs FP32
--------------------------------------------------------
  alpha 0.5: entail pearson 0.9810  max|d| 0.338  size 318 MB
  alpha 0.7: entail pearson 0.9957  max|d| 0.168  size 318 MB  (best)
  alpha 0.9: entail pearson 0.9863  max|d| 0.345  size 318 MB
ORT baselines (no SmoothQuant): dynamic 0.35 / static 0.29 / mixed 0.754 / wo4bit 0.90

Best: alpha 0.7 at 0.9957, 318 MB

## Distance-fidelity gate - the quality metric

Parity on raw entailment is a proxy; the real gate is whether the **source-conditioned distance reaches the same verdict** with the INT8 entailer. For the best alpha, rebuild `D_grd` per fixture under FP32 and INT8 (reranker premises fixed) and check the verdict is preserved: tier-mean ordering reproduced, severity regime matched, and per-document ordinality no worse than FP. The per-fixture `D_grd` Pearson is a supporting diagnostic - a derived L2 signature-distance amplifies small per-statement entailment deltas, so it sits below raw-entail parity even when the tier verdict is unchanged. This replaces the original recipe's grounding-stack macro-F1, which does not apply to a source-conditioned distance.

In [10]:
sig_q = signatures(best["ent"], best["con"])
dgrd_q = d_grd(sig_q)

labels = list(TIER)
dr = parity([dgrd_fp[l] for l in labels], [dgrd_q[l] for l in labels])
tm_fp, tm_q = tier_means(dgrd_fp), tier_means(dgrd_q)
v_fp, v_q = violations(dgrd_fp), violations(dgrd_q)
sev_fp, sev_q = severity(dgrd_fp), severity(dgrd_q)
order_fp = tm_fp["gold"] < tm_fp["adv1"] < tm_fp["adv2"]
order_q = tm_q["gold"] < tm_q["adv1"] < tm_q["adv2"]

rows = "\n".join(
    f"  {l:8s} [{TIER[l]:4s}]  FP [yellow]{dgrd_fp[l]:.4f}[/yellow]  INT8 [cyan]{dgrd_q[l]:.4f}[/cyan]  "
    f"[dim]d {abs(dgrd_fp[l] - dgrd_q[l]):.4f}[/dim]"
    for _, l, _ in DOCS)
rprint(f"""[bold cyan]Per-fixture D_grd - FP32 vs INT8 (alpha {best['alpha']})[/bold cyan]
[dim]{"-" * 56}[/dim]
{rows}

[bold]tier means[/bold]
""" + "\n".join(
    f"  {TIER_NAME[t]:18s}  FP [yellow]{tm_fp[t]:.4f}[/yellow]  INT8 [cyan]{tm_q[t]:.4f}[/cyan]"
    for t in TIERS))

ordering_ok = order_q == order_fp                  # INT8 reproduces FP's tier-ordering decision
severity_ok = sev_q == sev_fp                      # same Set 1 / Set 2 regime
ordinality_ok = v_q <= v_fp                        # INT8 no worse on per-document violations
verdict_ok = ordering_ok and severity_ok and ordinality_ok
dgrd_strong = dr["pearson"] >= TARGET_DGRD_R       # diagnostic only
rprint(f"""
[bold]D_grd parity[/bold] [dim](diagnostic)[/dim]: pearson [{'green' if dgrd_strong else 'yellow'}]{dr['pearson']:.4f}[/] """
f"""spearman {dr['spearman']:.4f}  max|d| {dr['max_abs']:.4f}
[bold]verdict preserved[/bold]: ordering [{'green' if ordering_ok else 'red'}]{order_q} (FP {order_fp})[/]  """
f"""severity [{'green' if severity_ok else 'red'}]{sev_q} (FP {sev_fp})[/]  """
f"""violations [{'green' if ordinality_ok else 'red'}]{v_q} <= {v_fp}[/]""")

Per-fixture D_grd - FP32 vs INT8 (alpha 0.7)
--------------------------------------------------------
  gold       FP 0.0000  INT8 0.0000  d 0.0000
  gold-2     FP 0.2246  INT8 0.2279  d 0.0033
  v1         FP 0.1770  INT8 0.1795  d 0.0025
  v2         FP 0.4241  INT8 0.4359  d 0.0118
  opus       FP 0.1360  INT8 0.1469  d 0.0109
  sonnet     FP 0.2393  INT8 0.2255  d 0.0138
  haiku      FP 0.3947  INT8 0.3917  d 0.0030
  adv1-a     FP 0.3473  INT8 0.3335  d 0.0138
  adv1-b     FP 0.2224  INT8 0.1959  d 0.0265
  adv2-a     FP 0.7828  INT8 0.7879  d 0.0051
  adv2-b     FP 0.4669  INT8 0.4648  d 0.0021

tier means
  gold (faithful)     FP 0.2280  INT8 0.2296
  Set 1 (info-loss)   FP 0.2848  INT8 0.2647
  Set 2 (info-noise)  FP 0.6248  INT8 0.6263

D_grd parity (diagnostic): pearson 0.9985 spearman 0.9909  max|d| 0.0265
verdict preserved: ordering True (FP True)  severity Set2>Set1 (FP Set2>Set1)  violations 6 <= 6

## Summary

Verdict for the source-conditioned CPU NLI: the best INT8 SmoothQuant entailer, its size, entailment parity, distance fidelity, and whether the `D_grd` verdict survives quantization. The deployable IR and metrics are written to disk.

In [11]:
parity_ok = best["pearson"] >= TARGET_PARITY
faithful = parity_ok and verdict_ok
verdict = ("FAITHFUL - INT8 SmoothQuant reproduces the source-conditioned distance verdict"
           if faithful else
           "below gate - keep mDeBERTa FP32 on CPU for the grounding axis")

# promote best IR to the deployable location, persist metrics (IR ignored by git, metrics tracked)
best_dir = Path(best["dir"])
for f in best_dir.iterdir():
    (WORK / f.name).write_bytes(f.read_bytes()) if f.is_file() else None
nli_tok.save_pretrained(WORK)
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.write_text(json.dumps({
    "model": MODEL_ID, "method": "openvino-nncf-smoothquant", "best_alpha": best["alpha"],
    "size_mb": best["size_mb"], "fp32_mb": fp32_mb,
    "entail_parity_pearson": best["pearson"], "dgrd_parity_pearson": dr["pearson"],
    "tier_means_fp": tm_fp, "tier_means_int8": tm_q,
    "violations_fp": v_fp, "violations_int8": v_q,
    "severity_fp": sev_fp, "severity_int8": sev_q,
    "ordering_fp": bool(order_fp), "ordering_int8": bool(order_q),
    "verdict_preserved": bool(verdict_ok), "faithful": bool(faithful),
    "calibration": "source top-3 joint premise x summary statement, exec-summary fixtures",
    "n_calib": N_CALIB,
}, indent=2))

rprint(f"""[bold]mDeBERTa-v3 INT8 on CPU - source-conditioned grounding result[/bold]
[dim]{"-" * 60}[/dim]
  Method: [cyan]OpenVINO / NNCF SmoothQuant[/cyan] (alpha [yellow]{best['alpha']}[/yellow])
  Size: [green]{best['size_mb']:.0f} MB[/green] [dim](FP32 {fp32_mb:.0f} MB, ~{fp32_mb / best['size_mb']:.1f}x smaller)[/dim]
  Entailment parity vs FP32: [{'green' if parity_ok else 'red'}]{best['pearson']:.4f}[/]
  D_grd parity vs FP32 [dim](diagnostic)[/dim]: [{'green' if dgrd_strong else 'yellow'}]{dr['pearson']:.4f}[/]
  Verdict preserved: [{'green' if verdict_ok else 'red'}]{verdict_ok}[/] [dim](ordering, severity, ordinality)[/dim]
  Calibration: [dim]source top-{TOP_K} joint premise x summary statement ({N_CALIB} pairs)[/dim]
  IR: [dim]{WORK}[/dim]

  [bold]{verdict}[/bold]
""")

mDeBERTa-v3 INT8 on CPU - source-conditioned grounding result
------------------------------------------------------------
  Method: OpenVINO / NNCF SmoothQuant (alpha 0.7)
  Size: 318 MB (FP32 1151 MB, ~3.6x smaller)
  Entailment parity vs FP32: 0.9957
  D_grd parity vs FP32 (diagnostic): 0.9985
  Verdict preserved: True (ordering, severity, ordinality)
  Calibration: source top-3 joint premise x summary statement (98 pairs)
  IR: /home/lab/workspace/learning/projects/docdistance/models/Q02-mdeberta-openvino-int8

  FAITHFUL - INT8 SmoothQuant reproduces the source-conditioned distance verdict